In [39]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder,StandardScaler

In [40]:
df = pd.read_csv('../0.dataset/dataset_penjualan_emas_kotor.csv')
df.head(5)

,Transaction_ID,Date,Customer_ID,Gold_Type,Karat,Weight_Gram,Price_Per_Gram,Total_Price,Payment_Method,Store_Location
0,TXN-0295,2024-04-21,CUST-480,Perhiasan,24,35.55,1370210,48710965,Kartu Kredit,JKT Pusat
1,TXN-0809,2024-03-09,CUST-373,Perhiasan,18,NaN,961081,11215815,Transfer Bank,Makassar
2,TXN-0364,2024-11-18,CUST-175,Perhiasan,24,49.59,1323332,65624033,QRIS,Bandung
3,TXN-0869,2025-09-30,CUST-313,Perhiasan,24,44.73,1352769,60509357,QRIS,Makassar
4,TXN-0326,2025-03-13,CUST-158,Koin,18,10.93,960581,10499150,Transfer Bank,Surabaya


In [41]:
df = df.dropna()
df.isnull().sum()

Transaction_ID    0
Date              0
Customer_ID       0
Gold_Type         0
Karat             0
Weight_Gram       0
Price_Per_Gram    0
Total_Price       0
Payment_Method    0
Store_Location    0
dtype: int64

In [42]:
df = df.drop_duplicates(keep='last')
df.duplicated().sum()

np.int64(0)

In [43]:
df = df.drop(columns=['Transaction_ID','Customer_ID','Date'])
df.head()

,Gold_Type,Karat,Weight_Gram,Price_Per_Gram,Total_Price,Payment_Method,Store_Location
0,Perhiasan,24,35.55,1370210,48710965,Kartu Kredit,JKT Pusat
2,Perhiasan,24,49.59,1323332,65624033,QRIS,Bandung
3,Perhiasan,24,44.73,1352769,60509357,QRIS,Makassar
5,Koin,24,38.79,1350607,52390045,Kartu Kredit,Makassar
6,Batangan,24,34.33,1362814,46785404,QRIS,Medan


In [44]:
colom_numeric = df.select_dtypes(include=np.number).columns
colom_categorical = df.select_dtypes(include='object').columns

In [45]:
df[colom_categorical] = df[colom_categorical].apply(lambda x:x.str.title())

In [46]:
label = LabelEncoder()
for col in colom_categorical:
    df[col] = label.fit_transform(df[col])

## 1.Feature Selection (Pemilihan Fitur)

### 1.Menghapus Fitur dengan Varians Rendah

In [ ]:
from sklearn.feature_selection import VarianceThreshold
df_target = df.copy()

colom_target = 'Gold_Type'
df_x = df_target.drop(columns=colom_target)
df_y = df_target[colom_target]

selector = VarianceThreshold(threshold=0.1)
selector.fit(df_x)

selected_features = df_x.columns[selector.get_support()]

df_x = df_x[selected_features]
kolom_numeric_tersisa = [col for col in selected_features if col in colom_numeric]

scaler = StandardScaler()
df_target = df_x.copy()
df_target[kolom_numeric_tersisa] = scaler.fit_transform(df_target[kolom_numeric_tersisa])
df_target['gold_target'] = df_y

df_target.head()

,Karat,Weight_Gram,Price_Per_Gram,Total_Price,Payment_Method,Store_Location,gold_target
0,1.117092,0.737917,1.387783,0.202386,0,2,2
2,1.117092,1.696567,1.105956,0.432261,1,0,2
3,1.117092,1.364727,1.282929,0.362745,1,3,2
5,1.117092,0.959144,1.269932,0.252390,0,3,1
6,1.117092,0.654615,1.343319,0.176215,1,4,0


In [50]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import VarianceThreshold

# --- [ PROSES PREPROCESSING ANDA - SUDAH OK ] ---
df = pd.read_csv('../0.dataset/dataset_penjualan_emas_kotor.csv')
df = df.dropna()
df = df.drop_duplicates(keep='last')
df = df.drop(columns=['Transaction_ID', 'Customer_ID', 'Date'])

colom_numeric = df.select_dtypes(include=np.number).columns
colom_categorical = df.select_dtypes(include='object').columns

# Merapikan teks menjadi format Title Case
df[colom_categorical] = df[colom_categorical].apply(lambda x: x.str.title())

# Encoding data kategori
label = LabelEncoder()
for col in colom_categorical:
    df[col] = label.fit_transform(df[col])


# --- [ PERBAIKAN DI SINI: FEATURE SELECTION ] ---

# 1. Tentukan nama kolom target Anda (Misal targetnya adalah 'Gold_Type')
# SILAKAN GANTI 'Gold_Type' dengan nama kolom target asli di dataset Anda
nama_target = 'Gold_Type' 

# 2. Pisahkan Fitur (X) dan Target (y) agar target aman dari seleksi
X = df.drop(columns=[nama_target])
y = df[nama_target]

# 3. Jalankan VarianceThreshold pada data ASLI (sebelum StandardScaler)
# Ini agar varians asli dari fitur numerik (gram, harga, dll) bisa dihitung secara adil
selector = VarianceThreshold(threshold=0.1)
selector.fit(X)

# Ambil nama-nama kolom fitur yang lolos seleksi
selected_features = X.columns[selector.get_support()]

# Filter X hanya menggunakan fitur yang lolos
X_selected = X[selected_features]

# 4. Lakukan StandardScaler HANYA pada fitur-fitur numerik yang LOLOS seleksi
# Kita cari tahu kolom numerik apa saja yang tersisa
kolom_numeric_tersisa = [col for col in selected_features if col in colom_numeric]

scaler = StandardScaler()
X_selected = X_selected.copy() # Menghindari SettingWithCopyWarning
X_selected[kolom_numeric_tersisa] = scaler.fit_transform(X_selected[kolom_numeric_tersisa])

# 5. Gabungkan kembali dengan Target jika ingin melihat hasil akhirnya dalam satu DataFrame
df_final = X_selected.copy()
df_final[nama_target] = y

print("Fitur yang berhasil dipertahankan:", list(selected_features))
df_final.head()

Fitur yang berhasil dipertahankan: ['Karat', 'Weight_Gram', 'Price_Per_Gram', 'Total_Price', 'Payment_Method', 'Store_Location']


,Karat,Weight_Gram,Price_Per_Gram,Total_Price,Payment_Method,Store_Location,Gold_Type
0,1.117092,0.737917,1.387783,0.202386,0,2,2
2,1.117092,1.696567,1.105956,0.432261,1,0,2
3,1.117092,1.364727,1.282929,0.362745,1,3,2
5,1.117092,0.959144,1.269932,0.252390,0,3,1
6,1.117092,0.654615,1.343319,0.176215,1,4,0
